In [ ]:
%matplotlib inline
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import seaborn as sns
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from implementation_heuristic import *

In [ ]:

PROBLEM = "JWST_INV"
udp = programmable_cubes_UDP(PROBLEM)
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)

In [ ]:
#NAME = "CORRECT_CHROMOSOME_OF_INVERSE_PROBLEM"
NAME = "jwst_chromosome_missing36"
chromosome = np.load(NAME + ".npy")
chromosome = np.array(chromosome,dtype=int)
print(udp.fitness(end_chromosome(chromosome)))
udp.plot("ensemble",types)
udp.plot("target",types)

In [ ]:
PROBLEM2 = "JWST"
udp2 = programmable_cubes_UDP(PROBLEM2)
udp2.fitness(np.array([-1]))
ti2 = udp2.initial_cube_types
ci2 = udp2.final_cube_positions
ct2 = udp2.target_cube_positions
tt2 = udp2.target_cube_types
types2 = np.arange(np.max(ti)+1)

In [ ]:
def invert_chromosome_better(udp:programmable_cubes_UDP,chromosome,same_ids = False):
    """
    expect even length 1D array
    """
    assert(chromosome.size % 2 == 0)
    inv_chromosome = []
    # extract moves and ids
    moves = np.array(chromosome[::-2],dtype=int)
    ids = np.array(chromosome[(len(chromosome)-2)::-2],dtype=int)
    print(ids)
    print(moves)
    # invert rotations
    moves = [inv_rot(moves[i]) for i in range(len(moves))]
    # convert ids
    # i is from target, id is from achieved, id -> i
    # udp is the inverse problem
    udp.fitness(end_chromosome(chromosome))

    achieved_config = udp.final_cube_positions
    avail_ids_achieved_config = np.ones(shape=len(achieved_config),dtype=bool)
    target_config = udp.target_cube_positions
    avail_ids_target_config = np.ones(shape=len(achieved_config),dtype=bool)

    id_conversion = {}
    for i in np.arange(len(target_config)):
        coord = target_config[i]
        id = contains_coord(achieved_config,coord)
        if id != -1:
            id_conversion[int(id)] = int(i)
            avail_ids_achieved_config[id] = False
            avail_ids_target_config[i] = False
    # randomly match the other cubes
    for i in np.arange(len(target_config)):
        if not avail_ids_achieved_config[i]:
            continue
        for j in np.arange(len(target_config)):
            if avail_ids_target_config[j]:
                id_conversion[i] = j
                avail_ids_achieved_config[i] = False
                avail_ids_target_config[j] = False
                break
    print(id_conversion)
    print(ids)

    mapped_ids = [id_conversion.get(int(_id),-1) for _id in ids]
    if same_ids:
        mapped_ids = ids
    #print(ids)
    #print(moves)
    if any(m == -1 for m in mapped_ids):
        raise RuntimeError(f"Unmapped ids after conversion: {mapped_ids}")

    # 
    inv_pairs = [[mapped_ids[i],moves[i]] for i in range(len(moves))]
    inv_chromosome = np.array(inv_pairs).flatten()
    print(id_conversion[61])
    return inv_chromosome



def invert_chromosome_same_problem(chromosome):
    ch = np.array(chromosome, dtype=int).flatten()
    pairs = ch.reshape(-1, 2)[::-1]                  # reversed rows [id,move]
    ids = pairs[:,0].astype(int).tolist()
    moves = [inv_rot(int(m)) for m in pairs[:,1].astype(int).tolist()]

    inv_pairs = np.column_stack([np.array(ids, dtype=int), np.array(moves, dtype=int)])
    return inv_pairs.flatten()


print(chromosome)
filtered_chromosome = filter_impossible_moves(udp,chromosome)
inv_chromosome = invert_chromosome_better(udp,filtered_chromosome,True)
#inv_chromosome = invert_chromosome_same_problem()
print(inv_chromosome)
# udp2.plot("ensemble",types2)
inv_chromosome_for_inverse_problem = invert_chromosome_better(udp,filtered_chromosome)
print(udp2.fitness(end_chromosome(inv_chromosome_for_inverse_problem)))
udp2.plot("ensemble",types2)
cubes = ProgrammableCubes(udp.target_cube_positions)

total_chromosome = np.concatenate([filtered_chromosome,inv_chromosome])
print(len(filtered_chromosome))
udp.fitness(end_chromosome(np.array([-1])),ci,True)
udp.plot("ensemble",types)
udp.fitness(format_chromosome(udp,filtered_chromosome),ci,True)
udp.plot("ensemble",types)
udp.fitness(end_chromosome(total_chromosome),ci,True)
udp.plot("ensemble",types)